# Lesson 02 Lab — 1T1C DRAM: Charge Sharing, Sensing, and Restore

**Puzzle:** If a DRAM cell is only one transistor and one tiny capacitor, how can a read recover a reliable bit without preserving the original charge?

This notebook retains one complete RTX 5090 execution.


## Why this matters

A 1T1C cell trades circuit area for a demanding read protocol. The wordline turns on an access transistor, the cell capacitor shares charge with a much larger bitline precharged near `VDD/2`, and a sense amplifier turns the small deviation into a full logic level. Because the access changes the cell charge, sensing is followed by restore; leakage later requires refresh.


## 0. Predict before running

1. Predict whether a stored 1 moves the bitline above or below precharge.
2. Predict how a 10× larger bitline capacitance changes the sensing margin.
3. Explain why the read is called destructive.

For each prediction, write the observation that would disprove it.


## 1. Theory and mechanism

Ignoring parasitics, shared voltage is `(Ccell·Vcell + Cbit·Vpre)/(Ccell + Cbit)`. The signal margin is the absolute deviation from precharge. Increasing bitline capacitance reduces that margin; lowering retained cell voltage does the same. The notebook sweeps both effects and records the restore target. This numerical result explains the mechanism, but it does not identify a proprietary DRAM timing or analog sense-amplifier design.

- Precharge creates a neutral reference near `VDD/2`.
- Charge sharing produces a small analog deviation before a digital bit exists.
- Read, sense, and restore are one protocol; omitting restore loses the state.


## 2. Trace the mechanism

### Mechanism map

```mermaid
flowchart LR
  A["bitline precharge"] --> B["wordline opens access transistor"]
  B --> C["cell and bitline share charge"]
  C --> D["sense amplifier resolves deviation"]
  D --> E["cell is restored"]
```


## 3. Inspect the visual boundary

![1T1C DRAM cell](../assets/1T1C_DRAM_Cell.png)

![DRAM read mechanism](../assets/visualizations/dram-1t1c-read-mechanism.png)

- [Interactive DRAM read visualization](../assets/visualizations/dram-1t1c-read-mechanism.html)

These are conceptual teaching diagrams. They explain the named data path and are not die-accurate schematics of a particular commercial GPU.


## 4. Inspect the execution environment

The next cell asserts CUDA, records GPU/PyTorch/CUDA identity, fixes the seed, and defines the common event-timing helpers.


In [1]:
LESSON_NO = 2
LESSON_TITLE = '1T1C DRAM: Charge Sharing, Sensing, and Restore'

from pathlib import Path
from collections import Counter, deque
import json, math, platform, statistics, sys, time

import torch
import torch.nn.functional as F

assert torch.cuda.is_available(), "Chapter 04 retained runs require a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260813 + LESSON_NO
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

major, minor = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
ENV = {
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    pos = (len(ordered) - 1) * q
    lo, hi = math.floor(pos), math.ceil(pos)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def cuda_samples(fn, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    samples = []
    for _ in range(repeats):
        start = torch.cuda.Event(enable_timing=True)
        stop = torch.cuda.Event(enable_timing=True)
        start.record()
        fn()
        stop.record()
        stop.synchronize()
        samples.append(float(start.elapsed_time(stop)))
    return samples

def summary(samples):
    return {
        "median_ms": statistics.median(samples),
        "p95_ms": percentile(samples, 0.95),
        "samples_ms": samples,
    }


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "seed": 20260815
}


## 5. Freeze the experiment

| Role | Frozen value |
|---|---|
| Baseline | fresh cell at 1.0 V with a 10× bitline-to-cell capacitance ratio |
| Candidate | larger bitlines and leaked cell voltage |
| Held constant | VDD, precharge voltage, and ideal charge conservation |
| Measurements | shared voltage, sensing margin, and margin loss |
| Evidence | `numerical-model` |

**Experiment:** Compute charge-sharing voltage and sensing margin over capacitance and retention sweeps.


## 6. Inspect the code

The experiment uses a small pure-Python function for the conservation equation, sweeps explicit values, and checks that a restored 1 returns to VDD. No latency number is manufactured from the model.

Do not run until the code matches the frozen table.


In [2]:
VDD = 1.0
VPRE = VDD / 2
C_CELL = 30e-15

def shared_voltage(v_cell, c_cell, v_bit, c_bit):
    return (v_cell * c_cell + v_bit * c_bit) / (c_cell + c_bit)

def margin_mv(v_cell, bitline_ratio):
    shared = shared_voltage(v_cell, C_CELL, VPRE, C_CELL * bitline_ratio)
    return abs(shared - VPRE) * 1e3

fresh = margin_mv(1.0, 10)
leaked = margin_mv(0.72, 10)
ratios = {str(r): margin_mv(1.0, r) for r in (5, 10, 20, 40, 80)}
retention = {str(v): margin_mv(v, 10) for v in (1.0, 0.9, 0.8, 0.72, 0.6)}
metrics = {
    "vdd_v": VDD,
    "precharge_v": VPRE,
    "cell_capacitance_f": C_CELL,
    "fresh_margin_mv": fresh,
    "leaked_margin_mv": leaked,
    "margin_retained": leaked / fresh,
    "restore_target_v": VDD,
    "bitline_ratio_sweep_margin_mv": ratios,
    "retention_sweep_margin_mv": retention,
}
analysis = (
    f"With a 10:1 bitline/cell capacitance ratio, the ideal fresh-cell deviation was "
    f"{fresh:.3f} mV and fell to {leaked:.3f} mV when retained cell voltage was 0.72 V. "
    "The sense amplifier and restore step are therefore part of the read contract."
)
print(json.dumps(metrics, indent=2))


{
  "vdd_v": 1.0,
  "precharge_v": 0.5,
  "cell_capacitance_f": 3e-14,
  "fresh_margin_mv": 45.45454545454541,
  "leaked_margin_mv": 20.000000000000018,
  "margin_retained": 0.44000000000000083,
  "restore_target_v": 1.0,
  "bitline_ratio_sweep_margin_mv": {
    "5": 83.33333333333337,
    "10": 45.45454545454541,
    "20": 23.809523809523835,
    "40": 12.195121951219413,
    "80": 6.172839506172867
  },
  "retention_sweep_margin_mv": {
    "1.0": 45.45454545454541,
    "0.9": 36.363636363636374,
    "0.8": 27.272727272727227,
    "0.72": 20.000000000000018,
    "0.6": 9.090909090909038
  }
}


## 7. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Fresh-cell margin | 45.4545 |
| Leaked-cell margin | 20.0000 |
| Margin retained | 44.00% |
| Restore target | 1.0000 |


## 8. Explain rather than overclaim

With a 10:1 bitline/cell capacitance ratio, the ideal fresh-cell deviation was 45.455 mV and fell to 20.000 mV when retained cell voltage was 0.72 V. The sense amplifier and restore step are therefore part of the read contract.

**Evidence boundary:** A transparent mechanism model executed. It establishes the stated relationship under printed assumptions, not native hardware latency, energy, or topology.


## 9. Write the canonical artifact

The next cell stores the environment, metrics, analysis, evidence label, and bounded conclusion, then prints the exact JSON.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 2, "title": '1T1C DRAM: Charge Sharing, Sensing, and Restore', "environment": ENV,
    "evidence_label": 'numerical-model', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Treat sensing margin as the bridge between capacitor physics and a reliable digital bit; restore and refresh are required parts of the storage contract.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 2,
  "title": "1T1C DRAM: Charge Sharing, Sensing, and Restore",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "seed": 20260815
  },
  "evidence_label": "numerical-model",
  "metrics": {
    "vdd_v": 1.0,
    "precharge_v": 0.5,
    "cell_capacitance_f": 3e-14,
    "fresh_margin_mv": 45.45454545454541,
    "leaked_margin_mv": 20.000000000000018,
    "margin_retained": 0.44000000000000083,
    "restore_target_v": 1.0,
    "bitline_ratio_sweep_margin_mv": {
      "5": 83.33333333333337,
      "10": 45.45454545454541,
      "20": 23.809523809523835,
      "40": 12.195121951219413,
      "80": 6.172839506172867
    },
    "retention_sweep_margin_mv": {
      "1.0": 45.45454545454541,
      "0.9": 36.363636363636374,
      "0.8": 27.272727272727227,
      "0.72": 20.000000000000018,
      "0.6": 9.090909090909038
    }
  },
  "analysis": "With a 10:

## 10. Make the decision

> Treat sensing margin as the bridge between capacitor physics and a reliable digital bit; restore and refresh are required parts of the storage contract.

**Failure analysis:** Parasitic capacitance, noise, temperature, variation, equalization, and sense-amplifier offset are omitted. The model is useful for direction, not for sign-off.


## 11. Extend the evidence

Add a noise and offset distribution, then estimate a margin-failure probability rather than reporting only the nominal voltage.

See [`README.md`](README.md) for the full explanation and references.
